# Notebook 012 — Volume-Confirmed Breakout, One Rule, Whole Basket


In [1]:
import sys

sys.path.insert(0, "..")
sys.path.insert(0, "tmp")
import json

TMP = "tmp"


def load(name):
    with open(f"{TMP}/{name}") as f:
        return json.load(f)

## Phase 0 — Universe, regime gates, and frozen thresholds

Pooled basket: 30 crypto perpetuals, 42 commodity-equity/ETF tickers (the
27 `*=F` FX/futures proxies excluded -- their yfinance volume is unreliable,
NEXT_PROMPT.md sec 1), and 16 databento futures products with OHLCV carried
through the roll via a new `commod_lib8.build_continuous_series_ohlcv`
(the existing `build_continuous_series` drops open/high/low/volume
entirely). 88 instruments total.

Thresholds are frozen once, before any pooled backtest runs, from each
instrument's own first 3 years of history (or its full history if
shorter) -- never re-tuned after seeing a result. All three are
scale-free (ATR multiples or a volume-ratio percentile), not %-of-price,
so a single set of numbers transfers across four very different asset
classes.


In [2]:
phase0 = load("phase_0_12_results.json")
print(f"Instruments: {phase0['n_instruments']} (total {phase0['total_instruments']})")
print(f"Regime gate open fraction: {phase0['regime_ok_frac']}")
print(f"Delisted crypto present: {phase0['delisted_crypto_present']}")
print(
    f"Frozen thresholds: {phase0['thresholds']['base_max_range_atr_mult']=}, "
    f"{phase0['thresholds']['prior_run_min_atr_mult']=}, "
    f"{phase0['thresholds']['vol_k']=}"
)
print(f"Calibration bars pooled: {phase0['thresholds']['n_calibration_bars']}")

Instruments: {'crypto': 30, 'equity': 42, 'futures': 16} (total 88)
Regime gate open fraction: {'crypto': 0.5408629839978765, 'equity': 0.5734606020460363, 'futures': 0.4386543515922797}
Delisted crypto present: ['FTTUSDT', 'LUNAUSDT']
Frozen thresholds: phase0['thresholds']['base_max_range_atr_mult']=2.8, phase0['thresholds']['prior_run_min_atr_mult']=4.5, phase0['thresholds']['vol_k']=1.39
Calibration bars pooled: {'base_range': 75581, 'prior_run': 72511, 'vol_ratio': 66645}


## Phase 1 — Gate VB pre-registration

Committed before the pooled backtest ran; Phase 2/3 assert against this
file programmatically rather than re-typing its criteria.


In [3]:
prereg = load("phase_1_12_preregistration.json")["gates"]["VB"]
print(f"n_trials: {prereg['n_trials']}")
print(f"fires_if: {prereg['fires_if']}")

n_trials: 12
fires_if: net Sharpe > 0 at every origin offset (0/7/14/21) on the volume-gated 1x-cost book AND paired block-bootstrap 95% CI on (volume-gated minus ungated) offset-0 1x-cost returns excludes zero AND DSR > 0.95 at n_trials=12 (4 offsets x 3 cost multipliers, single fixed rule and single fixed k -- no other parameter swept) AND cost stress at 3x shows a real, correctly-signed degradation AND the three-way risk gate / fail-closed regime gate carry over unchanged from 11a/11d


## Phase 2 — Gate VB: pooled book, volume-gated vs. the identical ungated control

The control is the byte-identical breakout rule (same signals, stops,
exits, costs, bars, regime gate) with only the volume condition switched
off -- the mechanism isolation NEXT_PROMPT.md sec 3 asks for.


In [4]:
phase2 = load("phase_2_12_results.json")
print("Sharpe (volume-gated, 1x cost) by offset:")
for off, s in phase2["sharpes_gated_1x_by_offset"].items():
    print(f"  {off}: {s:.4f}")
print(f"Positive every offset: {phase2['positive_every_offset']}")
print()
gvu = phase2["gated_vs_ungated_bootstrap_offset0_1x"]
print(
    f"Gated-minus-ungated delta (offset 0, 1x): point {gvu['delta_point']:.4f}, "
    f"95% CI [{gvu['delta_ci'][0]:.4f}, {gvu['delta_ci'][1]:.4f}], "
    f"excludes zero: {gvu['delta_excludes_zero']}"
)
print()
cs = phase2["cost_stress_1x_vs_3x_gated_offset0"]
print(
    f"Cost stress (1x vs 3x, gated, offset 0): delta {cs['delta_point']:.4f}, "
    f"95% CI [{cs['delta_ci'][0]:.4f}, {cs['delta_ci'][1]:.4f}], "
    f"correctly signed: {phase2['cost_stress_correctly_signed']}"
)
print()
print(
    f"Deflated Sharpe prob (n_trials={phase2['n_trials']}): {phase2['deflated_sharpe_prob']:.3f}"
)
print(f"DSR fires: {phase2['dsr_fires']}")
print(f"Gate VB fires: {phase2['gate_fires']}")
print(f"Fundable flag: {phase2['fundable_flag']}")
print()
print(f"n trades gated (offset 0, 1x): {phase2['n_trades_gated_offset0_1x']}")
print(f"n trades ungated (offset 0, 1x): {phase2['n_trades_ungated_offset0_1x']}")
print(
    f"Trade counts by asset class, gated: {phase2['trade_counts_by_asset_class_gated']}"
)
print(
    f"Trade counts by asset class, ungated: {phase2['trade_counts_by_asset_class_ungated']}"
)
print(
    f"Max drawdown (offset 0, 1x, gated): {phase2['max_drawdown_offset0_1x_gated']:.4f}"
)

Sharpe (volume-gated, 1x cost) by offset:
  offset_0: 0.1150
  offset_7: 0.1150
  offset_14: 0.1151
  offset_21: 0.1152
Positive every offset: True

Gated-minus-ungated delta (offset 0, 1x): point -0.1820, 95% CI [-1.4895, 0.8702], excludes zero: False

Cost stress (1x vs 3x, gated, offset 0): delta -0.1608, 95% CI [-0.1870, -0.1368], correctly signed: True

Deflated Sharpe prob (n_trials=12): 0.064
DSR fires: False
Gate VB fires: False
Fundable flag: False

n trades gated (offset 0, 1x): 406
n trades ungated (offset 0, 1x): 1105
Trade counts by asset class, gated: {'crypto': 8, 'equity': 383, 'futures': 15}
Trade counts by asset class, ungated: {'crypto': 15, 'equity': 936, 'futures': 154}
Max drawdown (offset 0, 1x, gated): -0.4257


## Phase 3 — Final gate table, cross-checked against pre-registration


In [5]:
phase3 = load("phase_3_12_results.json")
print(f"Gate VB fires: {phase3['fires']}")
print(f"Fundable: {phase3['fundable']}")
print("Legs:")
for k, v in phase3["legs"].items():
    print(f"  {k}: {v}")

Gate VB fires: False
Fundable: False
Legs:
  positive_every_offset: True
  gated_vs_ungated_ci_excludes_zero: False
  dsr_fires: False
  cost_stress_correctly_signed: True


## What this notebook establishes, plainly

Gate VB does not fire. Net Sharpe is positive at every offset on the
volume-gated book (+0.115 at 1x cost, essentially flat across offsets 0/7/
14/21 -- an artifact of stacking a small origin-offset shift on top of a
per-instrument calibration exclusion that already removes each
instrument's first ~3 years, so shifting the remaining start by a further
0/7/14/21 bars leaves an identical 406-trade book at 3-decimal precision;
see the results markdown for the full disclosure). But the volume filter
does not earn its keep: the gated-minus-ungated paired bootstrap CI
includes zero and the point estimate goes the wrong way (gated book's
pooled return is *lower* than the ungated control's, not higher), and the
Deflated Sharpe Ratio is far below the 0.95 bar even at the honestly small
n_trials=12. Cost stress is correctly signed and real, same as every
other notebook in this programme -- the null is not explained by an
inflated cost assumption. Trade counts are reported per asset class
because pooling here is not balanced: equities supply the large majority
of trades, and that imbalance is part of the finding, not hidden by it.
